# Client Fact Library — End-to-End Demo

This notebook demonstrates the full pipeline:
1. Crawl a mock local business website
2. Score pages by business importance
3. Extract typed facts via LLM (Google Gemini)
4. Embed facts locally (sentence-transformers)
5. Store facts in Qdrant
6. Query facts via semantic search

**To run in Google Colab:** File → Open in Colab, then set your `GOOGLE_API_KEY` below.

In [ ]:
# Install dependencies
!pip install httpx beautifulsoup4 lxml pydantic jinja2 sentence-transformers qdrant-client fastapi uvicorn google-generativeai -q

In [ ]:
import os

# Set your Google AI Studio API key (get one free at aistudio.google.com)
os.environ['LLM_MODE'] = 'direct'
os.environ['LLM_PROVIDER'] = 'google'
os.environ['LLM_MODEL_DIRECT'] = 'gemini-2.5-flash'
os.environ['GOOGLE_API_KEY'] = 'YOUR_GOOGLE_API_KEY_HERE'  # <-- replace this
os.environ['EMBEDDING_MODE'] = 'local'
os.environ['QDRANT_URL'] = ':memory:'  # in-memory Qdrant for demo

In [ ]:
# Clone the repo (skip if already cloned)
import subprocess

result = subprocess.run(['git', 'clone', 'https://github.com/your-org/client-fact-library'], capture_output=True)
if result.returncode == 0:
    %cd client-fact-library
print('Ready')

## Step 1 — Page Importance Scoring

In [ ]:
from crawler.page_scorer import PageScorer

scorer = PageScorer()
test_urls = [
    '/', '/services/', '/pricing/', '/about/', '/faq/',
    '/blog/how-to-brush', '/contact/', '/privacy/'
]

print('URL Path                  | Score')
print('-' * 35)
for url in test_urls:
    score = scorer.score_url(url)
    print(f'{url:<25} | {score}')

## Step 2 — LLM Fact Extraction

In [ ]:
from extractor.fact_extractor import FactExtractor
from extractor.llm_client import build_llm_client

sample_page = """
Welcome to Sunrise Dental — Austin's trusted family dental practice since 2005.
Led by Dr. Sarah Mitchell, DDS. Board-certified, 20+ years experience.

Services: cleanings from $99, teeth whitening $299, Invisalign from $3,800.
Hours: Mon-Fri 8am-6pm, Sat 9am-2pm. Emergency appointments available same-day.
Serving Austin, Round Rock, and Cedar Park.
"""

llm_client = build_llm_client()
extractor = FactExtractor(llm_client=llm_client)

facts = extractor.extract(
    page_text=sample_page,
    page_url='http://example.com/dental/',
    page_type='homepage',
    page_score=4,
    industry='dental',
)

print(f'Extracted {len(facts)} facts:\n')
for fact in facts:
    print(f'  [{fact.fact_type}] {fact.content} (confidence: {fact.confidence:.2f})')

## Step 3 — Embed Facts and Store in Qdrant

In [ ]:
from embedder.local_embedder import LocalEmbedder
from store.qdrant_store import QdrantStore

embedder = LocalEmbedder()
store = QdrantStore(url=':memory:')

for fact in facts:
    embed_text = f'{fact.fact_type}: {fact.content}'
    vector = embedder.embed(embed_text)
    store.upsert_fact(
        client_id='demo-dental',
        fact=fact,
        vector=vector,
        source_url='http://example.com/dental/',
        page_type='homepage',
        page_score=4,
        content_hash='demo-hash-v1',
    )

counts = store.get_fact_counts_by_type('demo-dental')
print('Facts stored by type:', counts)

## Step 4 — Semantic Search

In [ ]:
query = 'what are your prices for teeth whitening'
query_vector = embedder.embed(query)

results = store.search(
    client_id='demo-dental',
    query_vector=query_vector,
    limit=3,
    fact_type='pricing',  # filter by type
)

print(f'Top results for: "{query}"\n')
for r in results:
    print(f'  Score: {r["score"]:.3f} | [{r["fact_type"]}] {r["content"]}')

## Summary

This demo showed:
- **Page scoring**: Service pages score 5, legal/privacy pages score 0 (skipped)
- **Typed extraction**: LLM returns structured Pydantic objects (pricing, service, location...)
- **Fact-typed vectors**: Each fact is one Qdrant point with `fact_type` as a payload filter
- **Filtered retrieval**: `fact_type=pricing` returns only pricing facts, ranked by relevance

For production: swap `QDRANT_URL` to Qdrant Cloud, `LLM_MODE` to `portkey` with your config slug.